# ЛР 2. Проверяемый цифровой верстак

**Сущность:** Проверить интерпретатор, зависимость и контракт численного преобразования до химического конвейера.

**Данные:** molecules.csv; системные версии и контрольные значения концентрации.

**Workflow:** Интерпретатор → версии → hash входа → функция с единицами → отрицательные контроли.

Перед запуском прочитайте [guideline](../guides/lab02.md) и [теорию](../../Module_theory/Module%202_Setup%20Workharness.md).

## 1. Прогноз

Запишите ожидаемое поведение и отрицательный контроль. Какое изменение входа должно изменить результат, а какое — сохранить?

In [1]:
from pathlib import Path
import sys, json, math, csv, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# WHY: relative discovery makes the notebook portable across machines.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Labs" / "chemistry.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите notebook внутри репозитория курса")
sys.path.insert(0, str(ROOT / "Labs"))
DATA = ROOT / "Labs" / "datasets"
OUT = ROOT / "Labs" / "results" / "lab02"
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()


## 2. Разбор и исполнение

1. Запишите путь Python и версии, отличая среду выполнения от редактора.
2. Посчитайте SHA256 входа и pIC50 двумя эквивалентными способами.
3. Проверьте отрицательный/нулевой ввод и неизвестную единицу без молчаливого исправления.

Ожидаемое: 100 nM = 0.1 uM → 7; ноль/неизвестная единица отклоняются; версии и hash сохранены.

In [2]:
from chemistry import p_activity
import hashlib, platform, rdkit
passport = {"python": platform.python_version(), "executable": sys.executable,
            "rdkit": rdkit.__version__, "sha256": hashlib.sha256((DATA/"molecules.csv").read_bytes()).hexdigest()}
print(passport)
assert math.isclose(p_activity(100,"nM"),7)
assert math.isclose(p_activity(0.1,"uM"),7)
for value,unit in [(0,"nM"),(-1,"M"),(1,"unknown")]:
    try:
        p_activity(value,unit)
    except ValueError:
        continue
    raise AssertionError("Некорректный ввод принят")
artifact = passport


{'python': '3.12.10', 'executable': 'C:\\Users\\Miakhlov\\Documents\\GitHub\\Computational chemistry_course\\.venv-course\\Scripts\\python.exe', 'rdkit': '2026.09.1', 'sha256': 'f1904ebd8af5789d383f9593365bf5443abf7aa60106db0ecfb50d4fff628465'}


## 3. Проверяемая запись запуска

Benchmark фиксирует этот запуск. Время включает расчёт и создание артефактов, но не установку среды; это не сравнительный рейтинг оборудования.

In [3]:
elapsed = time.perf_counter()-started
run_record = {"lab": 2, "calculation_wall_seconds": elapsed, "result": artifact}
(OUT/"benchmark.json").write_text(json.dumps(run_record,ensure_ascii=False,indent=2),encoding="utf-8")
print("Запуск сохранён:",OUT/"benchmark.json")


Запуск сохранён: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\Labs\results\lab02\benchmark.json


## 4. Самостоятельная тренировка

1. Допишите перевод uM в M без использования готовой функции.
2. Объясните ошибку импорта при двух Python.
3. Измените байт копии CSV и сравните hash.

[Ответы после попытки](../solutions/lab02.md). Вернитесь к первому заданию через неделю.

## 5. Ваш научный вывод

1. Какой результат получен?
2. Какой вывод из него пока не следует?
3. Какой следующий эксперимент или контроль нужен?

Сохраните свой вариант notebook отдельно; исходный пример должен остаться воспроизводимым.